# 小望遠鏡の実測データから 50cm 望遠鏡での写りを予測する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hsameshima/SmallTelescope_test01/blob/main/2026/notebooks/m57_practicum.ipynb)

2026年10月8日にミューロン210（口径 210 mm）＋ ZWO ASI183MM で撮った M57（環状星雲）のデータから、同じカメラを国立天文台の 50cm 公開望遠鏡に付けたときのピクセルスケール・信号・S/N を予測し、本番の観測スケジュールを作ります。

**最初にやること**：メニューの「ファイル → ドライブにコピーを保存」で自分用のコピーを作ってください。コピーしないと作業内容が保存されません。

**進め方**：上のセルから順に実行します。`# TODO` と書かれたセルは自分でコードを書き、「答え」のセルには文章で答えます。提出は「ファイル → ダウンロード → .ipynb をダウンロード」で行います。

## 0. 準備（実行するだけ）

データをダウンロードし、解析に使う道具（関数）を読み込みます。セッションが切れたら、ここからやり直してください。

In [ ]:
import os
REPO = "SmallTelescope_test01"
if not os.path.exists(REPO):
    !git clone --depth 1 --branch main https://github.com/hsameshima/SmallTelescope_test01.git
YEAR = "2026"                                   # 実習の年度（リポジトリのフォルダ名）
DATA = os.path.join(REPO, YEAR, "data", "20261008")
print(len(os.listdir(DATA)), "files in", DATA)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from scipy import ndimage as ndi

plt.rcParams["figure.dpi"] = 100

def files(filt):
    """フィルター名（'Ha', 'O3', 'S2', 'R', 'G', 'B', 'Dark'）のファイル一覧"""
    return sorted(glob.glob(os.path.join(DATA, f"*filter-{filt}_*.fit")))

def load(path):
    """FITS を読み込んで float の 2 次元配列で返す（単位 ADU）"""
    return fits.getdata(path).astype(float)

def mad_std(a):
    """中央値絶対偏差から求めた標準偏差。星やホットピクセルに強い"""
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return 1.4826 * np.median(np.abs(a - np.median(a)))

def show(img, x=None, y=None, size=None, ax=None, title=None, marks=()):
    """画像を表示する（左下が原点）。x, y, size を与えるとその周りを拡大。
    marks に [(x, y), ...] を与えると円で印を付ける"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    if size is None:
        vmin, vmax = ZScaleInterval().get_limits(img[np.isfinite(img)])
    else:
        cut = img[max(int(y - size), 0):int(y + size), max(int(x - size), 0):int(x + size)]
        vmin, vmax = np.nanpercentile(cut, [5, 99.5])
    ax.imshow(img, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    if size is not None:
        ax.set_xlim(x - size, x + size); ax.set_ylim(y - size, y + size)
    for (mx, my) in marks:
        ax.plot(mx, my, "o", mfc="none", mec="tab:orange", ms=14, mew=1.5)
    ax.set_xlabel("x [pix]"); ax.set_ylabel("y [pix]")
    if title: ax.set_title(title)
    return ax

def centroid(img, x0, y0, r=8, n_iter=5):
    """(x0, y0) 付近にある星（または天体）の重心 (x, y) を返す"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    x, y = float(x0), float(y0)
    for _ in range(n_iter):
        m = (xx - x)**2 + (yy - y)**2 <= r**2
        w = img[m] - np.median(img[(xx - x)**2 + (yy - y)**2 <= (2*r)**2])
        w = np.clip(w, 0, None)
        x, y = (xx[m] * w).sum() / w.sum(), (yy[m] * w).sum() / w.sum()
    return x, y

def aperture_mean(img, x, y, r=3):
    """(x, y) を中心とする半径 r 画素の円の中の平均値"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    return np.nanmean(img[(xx - x)**2 + (yy - y)**2 <= r**2])

def sky_level(img, xc, yc, r_in=90, r_out=140):
    """(xc, yc) の周りのリング（半径 r_in〜r_out 画素）の中央値 = 背景のレベル"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    d2 = (xx - xc)**2 + (yy - yc)**2
    return np.nanmedian(img[(d2 >= r_in**2) & (d2 <= r_out**2)])

def line_profile(img, x0, y0, angle_deg, rmax=80, width=2):
    """(x0, y0) を通り、x 軸から angle_deg 傾いた直線に沿った断面。
    r（-rmax〜+rmax 画素、0.25 画素刻み）と、幅 ±width 画素で平均した値を返す"""
    a = np.radians(angle_deg)
    r = np.arange(-rmax, rmax + 0.01, 0.25)
    prof = []
    for off in np.arange(-width, width + 0.01, 1.0):
        ys = y0 + r * np.sin(a) + off * np.cos(a)
        xs = x0 + r * np.cos(a) - off * np.sin(a)
        prof.append(ndi.map_coordinates(img, [ys, xs], order=1))
    return r, np.mean(prof, axis=0)

def _stars(img, nmax=40):
    d = img - ndi.median_filter(img, size=25)
    s = mad_std(d)
    lab, n = ndi.label(ndi.gaussian_filter(d, 1.5) > 8 * s)
    idx = np.arange(1, n + 1)
    area = ndi.sum(np.ones_like(d), lab, idx)
    flux = ndi.sum(d, lab, idx)
    com = np.array(ndi.center_of_mass(d, lab, idx))
    ok = area >= 6
    return com[ok][np.argsort(-flux[ok])[:nmax]]

def _shift(ref_stars, img):
    d = (ref_stars[:, None, :] - _stars(img)[None, :, :]).reshape(-1, 2)
    e = np.arange(-60.5, 61)
    H, _, _ = np.histogram2d(d[:, 0], d[:, 1], bins=[e, e])
    j, i = np.unravel_index(H.argmax(), H.shape)
    sel = (np.abs(d[:, 0] - (e[j] + .5)) < 1.5) & (np.abs(d[:, 1] - (e[i] + .5)) < 1.5)
    return np.median(d[sel], axis=0)

MASTER_DARK = np.mean([load(f) for f in files("Dark")], axis=0)
_REF_STARS = _stars(load(files("G")[0]) - MASTER_DARK)

def stack(filt, flat=None):
    """そのフィルターの全フレームをダーク減算し、G バンド 1 枚目に位置を合わせて平均する。
    flat にフラット（masterflat の結果）を与えると、位置を合わせる前にフラットで割る。
    単位は ADU / 画素 / 60 秒（1 枚あたり）。端の欠けた部分は NaN"""
    out = []
    for f in files(filt):
        d = load(f) - MASTER_DARK
        if flat is not None:
            d = d / flat
        dy, dx = _shift(_REF_STARS, d)
        out.append(ndi.shift(d, (dy, dx), order=3, mode="constant", cval=np.nan))
    return np.mean(out, axis=0)

FLAT_DIR = os.path.join(REPO, YEAR, "data", "20261008_flat")

def masterflat(filt):
    """10/8 の日没前の空で作ったマスターフラット（画像の中央を 1 に規格化）。filt は 'R', 'G', 'B', 'Ha', 'O3', 'S2'"""
    return fits.getdata(os.path.join(FLAT_DIR, f"masterflat_{filt}.fits"), ext=1).astype(float)

def fwhm(img, x0, y0, r=10):
    """(x0, y0) 付近の星に 2 次元ガウス関数を当てはめ、(FWHM [pix], 縦横の比, ピーク [ADU]) を返す。
    FWHM は星像が伸びた向きとそれに直交する向きの幾何平均。縦横の比が 1 より大きいほど星像が伸びている"""
    from scipy.optimize import curve_fit
    x0, y0 = centroid(img, x0, y0)
    i, j = int(round(y0)), int(round(x0))
    cut = img[i - r:i + r + 1, j - r:j + r + 1]
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    def g(c, a, x, y, sx, sy, t, b):
        X, Y = c
        u = (X - x) * np.cos(t) + (Y - y) * np.sin(t)
        v = -(X - x) * np.sin(t) + (Y - y) * np.cos(t)
        return (a * np.exp(-u**2 / (2 * sx**2) - v**2 / (2 * sy**2)) + b).ravel()
    b0 = np.median(cut)
    p, _ = curve_fit(g, (xx, yy), cut.ravel(), p0=[cut.max() - b0, x0 - j, y0 - i, 1.5, 1.5, 0.0, b0], maxfev=5000)
    sx, sy = abs(p[3]), abs(p[4])
    return 2.3548 * np.sqrt(sx * sy), max(sx, sy) / min(sx, sy), float(cut.max())

print("準備完了")

## 予想を書き残す

スライドで見た HST の M57 の画像（青 [O III]、緑 Hα、赤 [N II]）について、どの色が内側に、どの色が外側に広がっているかを書いてください。理由は書かなくてかまいません。3日目の講義で、自分のデータを使って確かめます。

**予想：**

（ここに書く）

## 課題1 ピクセルスケールと視野

位置が分かっている星 2 個の間隔からピクセルスケールを求め、ミューロン210 の実際の焦点距離と、50cm での写り方を計算します。

| 星（Tycho-2） | 赤経 α（°） | 赤緯 δ（°） | V 等級 | 画像上のおよその位置 |
| --- | --- | --- | --- | --- |
| TYC 2642-509-1 | 283.29843 | +33.05353 | 11.1 | 右上寄り（x ≈ 886, y ≈ 807） |
| TYC 2643-1347-1 | 283.51852 | +32.88811 | 11.2 | 左下の隅付近（x ≈ 70, y ≈ 30） |

座標は Gaia DR3（元期 2016.0）の値です。

まず G バンドの 1 枚目を表示して、2 つの星を確かめます。

In [ ]:
g1 = load(files("G")[0]) - MASTER_DARK
print(g1.shape, "(縦, 横) 画素")
show(g1, title="G band, frame 1", marks=[(886, 807), (70, 30)]);

### 1-1 星の重心を測る

`centroid(画像, x の目安, y の目安)` で重心を求めてください。拡大表示（`show(g1, x, y, size=20)`）で星の中心に印が来ているかも確かめます。

In [ ]:
# TODO: 2 つの星の重心 (x1, y1), (x2, y2) を求める
x1, y1 = ...
x2, y2 = ...
print(x1, y1, x2, y2)

### 1-2 角距離とピクセルスケール

2 星の角距離 θ は次の式で求まります（α, δ はラジアンに直して使う）。

$$\cos\theta = \sin\delta_1 \sin\delta_2 + \cos\delta_1 \cos\delta_2 \cos(\alpha_1 - \alpha_2)$$

θ を秒角（″）にし、画像上の距離（画素）で割るとピクセルスケール s（″/pix）になります。

In [ ]:
ra1, dec1 = 283.29843, 33.05353
ra2, dec2 = 283.51852, 32.88811

# TODO: 角距離 theta（秒角）、画像上の距離 d_pix（画素）、ピクセルスケール s（″/pix）
theta = ...
d_pix = ...
s = ...
print(theta, d_pix, s)

### 1-3 焦点距離と F 値、50cm でのピクセルスケールと視野

Bin4 の画素サイズは p = 2.4 µm × 4 = 9.6 µm です。焦点距離は f = 206265 × p / s で求まります。

50cm 公開望遠鏡は口径 500 mm、焦点距離 6030 mm です。画像の大きさは 1368 × 918 画素です。

In [ ]:
p_mm = 9.6e-3      # 画素サイズ [mm]
nx, ny = 1368, 918

# TODO: ミューロン210 の焦点距離 f_mew [mm] と F 値 F_mew（口径 210 mm）
f_mew = ...
F_mew = ...

# TODO: 50cm のピクセルスケール s_50 [″/pix] と F 値 F_50
s_50 = ...
F_50 = ...

# TODO: 両方の視野 [′]（横 × 縦）
fov_mew = ...
fov_50 = ...

**答え：**

（ここに書く）

### 1-4 M57 の外縁の大きさ

HST の観測では、M57 の電離ガスの外縁は長半径 約 42″、短半径 約 30″ です。

1. 42″ がミューロン210 と 50cm でそれぞれ何画素になるか計算してください。
2. Hα 画像で M57 の中心を求め、長軸に沿った断面（`line_profile`）を描いて、外側で強度がピークの半分に落ちる半径を読み取ってください。断面の角度を変えて、外縁がいちばん遠くなる向きが長軸です。

In [ ]:
ha = stack("Ha")
show(ha, 745, 445, 100, title="H-alpha (+[N II]), 2 x 60 s");

In [ ]:
# TODO: M57 の中心 (xc, yc) を求める（目で見た中心を目安に centroid(ha, x, y, r=60)。r はリング全体が入る大きさにする）
xc, yc = ...

# TODO: 42″ が何画素か（ミューロン210, 50cm）
r42_mew = ...
r42_50 = ...

# TODO: 角度を変えながら断面を描き、外側の半値半径を読み取る
r, prof = line_profile(ha, xc, yc, angle_deg=0)
plt.plot(r, prof); plt.xlabel("distance from centre [pix]"); plt.ylabel("ADU / pix / 60 s");

**答え：**

（ここに書く）

### 1-5 星像の大きさ（シーイング）を測る

星の像は、大気のゆらぎ（シーイング）や追尾のずれ、ピントのずれでぼけて、ある幅に広がります。この幅を、明るさがピークの半分になる幅（半値全幅、FWHM）で表します。

`fwhm(画像, x の目安, y の目安)` は、星に 2 次元のガウス関数を当てはめて、FWHM（画素）、縦横の比、ピークの値を返します。

1. G の 1 枚目で、星を 5 個以上選んで FWHM を測り、中央値をピクセルスケール s をかけて秒角（″）に直してください。ピークが 60000 ADU を超える星は飽和しているかもしれないので使いません。
2. G の 2 枚目（1 分後に撮ったもの）でも同じ星を測り、1 枚目と比べてください。
3. 測った星像の幅は、50cm（1-3 のピクセルスケール）では何画素になりますか。

In [ ]:
# 星の候補（明るい順。M57 の近くと画像の端は除く）
cand = [(x, y) for y, x in _stars(g1, nmax=30)
        if np.hypot(x - xc, y - yc) > 100 and 15 < x < g1.shape[1] - 15 and 15 < y < g1.shape[0] - 15]
print(np.round(cand[:10], 1))
fw, q, pk = fwhm(g1, *cand[0])
print(f"例: FWHM = {fw:.2f} pix、縦横の比 {q:.2f}、ピーク {pk:.0f} ADU")

g2 = load(files("G")[1]) - MASTER_DARK     # G の 2 枚目

# TODO: 5 個以上の星で FWHM を測り、中央値を画素と秒角（× s）で出す。g2 でも同じ星を測って比べる
fwhm_arc = ...       # 星像の幅 [″]（1 枚目の中央値）

# TODO: その幅は 50cm（s_50）で何画素か

**答え：**（星像の幅は何″か。1 枚目と 2 枚目で違ったら、その原因として何が考えられるか〔シーイングの変化、追尾のずれ、ピント〕。縦横の比も手がかりになる）

（ここに書く）

## 課題2 信号とノイズの実測

ミューロン210 での「60 秒・1 枚・1 画素あたり」の信号とノイズを実測します。

### 2-1 1 枚あたりのノイズ

同じ条件の 2 枚の差をとると、画素ごとに固定したパターン（ホットピクセルなど）が消え、ランダムなノイズだけが残ります。差の標準偏差は 1 枚のノイズの √2 倍です。ダークと各フィルターについて、1 枚のノイズ σ と、平均レベル（2 枚の平均の中央値）を求めてください。標準偏差には `mad_std` を使います。

In [ ]:
filters = ["Dark", "S2", "Ha", "O3", "R", "G", "B"]
level, sigma = {}, {}
for f in filters:
    a, b = [load(x) for x in files(f)]
    # TODO: 平均レベル level[f] と 1 枚のノイズ sigma[f]
    level[f] = ...
    sigma[f] = ...
    print(f, level[f], sigma[f])

### 2-2 ゲイン

光子（電子）の数 N はポアソン分布に従うので、分散は N に等しくなります。信号を ADU で表すと、1 ADU が何電子にあたるかをゲイン g（e⁻/ADU）とすると、平均 S（ADU）は N／g、分散は N／g² なので、σ² ＝ S／g になります。横軸に平均レベル、縦軸に σ² をとって直線を当てはめ（フォトン・トランスファー・カーブ）、傾き 1／g からゲイン g を求めてください。

In [ ]:
# TODO: level と sigma**2 をプロットし、np.polyfit で直線を当てはめる。傾きは 1/g
g = ...
print("g =", g, "e-/ADU")

**答え：**

（ここに書く）

### 2-3 M57 のリングの信号

各輝線の画像（2 枚を位置合わせして平均したもの）から背景を引き、リング上に半径 3 画素の開口を 4 か所以上置いて、平均の信号 S（ADU / pix / 60 s）とそのばらつきを求めてください。開口の位置は画像を拡大して自分で決めます。

In [ ]:
img = {f: stack(f) for f in ["Ha", "O3", "S2"]}
for f in img:
    img[f] = img[f] - sky_level(img[f], xc, yc)     # 背景を引く
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, f in zip(axs, img):
    show(img[f], xc, yc, 70, ax=ax, title=f)

In [ ]:
# TODO: リング上の開口の位置を決め、各輝線の平均信号 S[f] とばらつきを求める
positions = [(..., ...), (..., ...), (..., ...), (..., ...)]
S = {}
for f in img:
    vals = [aperture_mean(img[f], x, y, r=3) for x, y in positions]
    S[f] = np.mean(vals)
    print(f, S[f], np.std(vals))

### 2-4 1 枚・1 画素あたりの S/N

$$\left(\frac{S}{N}\right)_1 = \frac{S}{\sqrt{\sigma_{\rm bg}^2 + S / g}}$$

σ_bg は 2-1 で求めたそのフィルターの 1 枚のノイズ、S／g は天体そのものの光子ノイズの分散（ADU² 単位）です。

In [ ]:
# TODO: 各輝線の 1 枚・1 画素あたりの S/N
snr1_mew = {}

**答え：**

（ここに書く）

## 課題3 50cm での信号の換算

面輝度 Σ の天体を、口径 D・焦点距離 f の望遠鏡と画素サイズ p のカメラで撮るとき、1 画素が受ける光量は「集光面積」と「1 画素が見込む空の立体角」の積に比例します。

$$N_{\rm pix} \propto \Sigma \times \frac{\pi D^2}{4} \times \left(\frac{p}{f}\right)^2 \propto \Sigma\,\frac{p^2}{F^2}, \qquad F = \frac{f}{D}$$

1. 1 画素あたりの信号の比（50cm / ミューロン210）を計算してください。
2. 同じ角度面積（1″ × 1″）あたりの光子数の比を計算してください。
3. 星（点源）の総光量の比を計算してください。
4. 1-5 で測った星像の幅（FWHM）は、ミューロン210 と 50cm でそれぞれ何画素ですか（1-3 のピクセルスケールを使う）。3 の総光量の比と合わせて、星の中心の画素の明るさ（ピーク）の比を見積もり、どちらの望遠鏡で星が飽和しやすいか考えてください。
5. 夜空の背景光も「広がった光源」です。50cm で 1 画素あたりの背景光はどう変わりますか。課題2で判定したノイズの内訳は変わるでしょうか。

In [ ]:
# TODO: 比を計算する
ratio_pix = ...     # 1 画素あたり
ratio_area = ...    # 同じ角度面積あたり
ratio_star = ...    # 星の総光量

# TODO: 1-5 で測った星像の幅 fwhm_arc が何画素か（ミューロン210, 50cm）と、星のピークの比
seeing_pix_mew = ...
seeing_pix_50 = ...
ratio_peak = ...

**答え：**

（ここに書く）

## 課題4 S/N と必要積分時間の予測

カメラ・露出 60 秒・センサー温度は 10/8 と同じとし、背景ノイズ σ_bg は 1 画素あたり変わらないとします。S/N は枚数の平方根に比例して伸びます。

$$n_{\rm frame} = \left(\frac{(S/N)_{\rm goal}}{(S/N)_1}\right)^2$$

1. 50cm・Bin4 での 1 枚・1 画素あたりの S/N を各輝線で予測してください。
2. 3 × 3 画素にまとめた場合（約 1″/pix）の S/N を計算してください。まとめると信号は 9 倍、分散も 9 倍になります。1-5 で測った星像の幅は、まとめた後に何画素になりますか。星像の形を保つには、幅に 2〜3 画素以上あるのが目安です。3 × 3 にまとめてよいか、4 × 4 ではどうかも考えてください。
3. 3 色合成用に、リング上で 1 画素 S/N = 20 にするのに必要な枚数を、Bin4 のままの場合と 3 × 3 にまとめた場合で求めてください。
4. 動径分布用に、方向ごとの区画で S/N = 10 にするには何枚必要か見積もってください。区画は、外縁（明るさがリングのピークの半分になる位置）に置いた、動径方向 1″ × それに直交する方向 3″ の長方形とします。区画に入る画素数は自分で見積もり、外縁の明るさはリングのピークの半分とします。（楕円環で全周を平均すると、向きによる外縁の違いが見えなくなるので、向きごとに測ります）

In [ ]:
# TODO
S_50 = {}        # 50cm での信号 [ADU/pix/60s]
snr1_50 = {}     # 50cm・Bin4 の 1 枚・1 画素 S/N
snr1_50_b3 = {}  # 3x3 にまとめた場合
n_bin4 = {}      # S/N 20 に必要な枚数（Bin4）
n_b3 = {}        # S/N 20 に必要な枚数（3x3）

**答え：**

（ここに書く）

## 夕方のデータを読み込む（晴れ・曇り共通）

今夜撮った FITS ファイルを 1 つの zip にまとめ、下のセルを実行してアップロードします（FITS をそのまま複数選んでもかまいません）。何も選ばずに「キャンセル」すると、練習用に 10/8 のデータを使います。

ファイルの種類（Light / Dark）とフィルターはファイル名から、露出時間・センサー温度・撮影時刻は FITS ヘッダーから読みます。表の `jst` は撮影開始の日本時間です。

In [ ]:
import re, zipfile
import pandas as pd

TONIGHT = "tonight"
os.makedirs(TONIGHT, exist_ok=True)
try:
    from google.colab import files as colab_files      # Colab のアップロード機能
    for name in colab_files.upload():                  # ← ここで zip（または FITS）を選ぶ
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(name).extractall(TONIGHT)
        elif name.lower().endswith((".fit", ".fits")):
            os.replace(name, os.path.join(TONIGHT, name))
except ImportError:
    print("Colab 以外で動かすときは、tonight フォルダに FITS を置いてください")

_NAME = re.compile(r"^(?P<kind>[A-Za-z]+)_.*?_filter-(?P<filt>[^_]+)_")

def catalog(paths):
    """FITS の一覧表を作る"""
    rows = []
    for p in paths:
        m = _NAME.match(os.path.basename(p))
        h = fits.getheader(p)
        start = pd.to_datetime(h.get("DATE-STA", h.get("DATE-OBS")))
        rows.append(dict(kind=m["kind"] if m else "?", filt=m["filt"] if m else "?",
                         exp=float(h.get("EXPTIME", np.nan)), temp=float(h.get("CCD-TEMP", np.nan)),
                         start=start, jst=(start + pd.Timedelta(hours=9)).strftime("%H:%M:%S"),
                         name=os.path.basename(p), path=p))
    return pd.DataFrame(rows).sort_values("start").reset_index(drop=True)

paths_t = sorted(glob.glob(os.path.join(TONIGHT, "**", "*.fit*"), recursive=True))
if not paths_t:
    paths_t = sorted(glob.glob(os.path.join(DATA, "*.fit")))
    print("今夜のデータがないので、10/8 のデータで練習します")
T = catalog(paths_t)
T[["kind", "filt", "exp", "temp", "jst", "name"]]

In [ ]:
def pick(kind=None, filt=None, exp=None, after=None, before=None):
    """条件に合うファイルのパスのリスト。after / before は "17:40" のような日本時間（撮影開始で判定）"""
    m = np.ones(len(T), bool)
    if kind is not None:   m &= (T.kind == kind).values
    if filt is not None:   m &= (T.filt == filt).values
    if exp is not None:    m &= np.isclose(T.exp.values, exp)
    if after is not None:  m &= (T.jst >= after).values
    if before is not None: m &= (T.jst < before).values
    return list(T.path[m])

def pair_stats(paths, box=None):
    """同じ条件の 2 枚から、平均レベル（中央値, ADU）と 1 枚のノイズ σ（ADU）を返す。
    box=(x0, x1, y0, y1) を与えると、その範囲だけで測る"""
    a, b = load(paths[0]), load(paths[1])
    if box is not None:
        sl = (slice(box[2], box[3]), slice(box[0], box[1]))
        a, b = a[sl], b[sl]
    return np.median((a + b) / 2), mad_std(a - b) / np.sqrt(2)

def bg_noise(paths, xc, yc, r_in=90, r_out=140):
    """2 枚の差から、天体の周りのリング（半径 r_in〜r_out 画素）での 1 枚のノイズ σ_bg を返す"""
    a, b = load(paths[0]), load(paths[1])
    yy, xx = np.mgrid[:a.shape[0], :a.shape[1]]
    d2 = (xx - xc)**2 + (yy - yc)**2
    m = (d2 >= r_in**2) & (d2 <= r_out**2)
    return mad_std((a - b)[m]) / np.sqrt(2)

def stack_list(paths, dark=0.0, ref=None):
    """ファイルのリストからダークを引き、星の位置を合わせて平均する（ADU / pix / 1 枚）。
    ref に星の位置のリスト（_stars の結果）を与えると、それに合わせる。与えなければ 1 枚目に合わせる"""
    out = []
    for p in paths:
        d = load(p) - dark
        if ref is None:
            ref = _stars(d)
            out.append(d)
            continue
        try:
            dy, dx = _shift(ref, d)
        except Exception:
            dy, dx = 0.0, 0.0
        out.append(ndi.shift(d, (dy, dx), order=3, mode="constant", cval=np.nan))
    return np.mean(out, axis=0)

print("夕方の道具の準備完了")

## 夕方A（晴れの場合）　テスト露光で予測を確かめる

今夜撮った M57 の 1 枚と、10/8 の 1 枚を比べます。望遠鏡とカメラは 10/8 と同じなので、違いが出たらその原因を考えます。

### A-1 今夜のダーク

今夜の 60 秒のダーク 2 枚から 1 枚のノイズを求め、10/8 の値（課題2の `sigma["Dark"]`）と比べてください。センサー温度は表の `temp` 列にあります。

In [ ]:
dark60_t = pick("Dark", exp=60)
print(len(dark60_t), "frames")

# TODO: 今夜のダークの平均レベル lev_dark_t と 1 枚のノイズ sig_dark_t（pair_stats を使う）、センサー温度
lev_dark_t, sig_dark_t = ...
print("今夜:", lev_dark_t, sig_dark_t, "  10/8:", level["Dark"], sigma["Dark"])

### A-2 リングの信号と 1 枚の S/N

今夜の 60 秒の Hα・[O III]・[S II] を 2 枚ずつ重ね、課題2と同じ方法でリングの信号 S と 1 枚・1 画素の S/N を求めて、10/8 の値（`S`、`snr1_mew`）と比べます。

M57 の写る位置は 10/8 とずれているので、中心を測り直し、課題2で決めた開口を「中心からの位置」として移します。画像の向きが 10/8 と変わっていないかも、画像を見て確かめてください。

トワイライトフラットも同じフィルター名で保存されているので、`SKY_AFTER` にテスト露光を撮り始めた時刻を入れて、それより後のフレームだけを使います。

In [ ]:
SKY_AFTER = None        # ← テスト露光を撮り始めた日本時間（例: "17:40"）。それより前のフラットを除く

MASTER_DARK_T = np.mean([load(f) for f in dark60_t], axis=0) if dark60_t else 0.0
REF_T = _stars(load(pick("Light", "Ha", 60, after=SKY_AFTER)[0]) - MASTER_DARK_T)   # 全フィルターを Hα の 1 枚目に合わせる
img_t = {f: stack_list(pick("Light", f, 60, after=SKY_AFTER), MASTER_DARK_T, ref=REF_T)
         for f in ["Ha", "O3", "S2"] if len(pick("Light", f, 60, after=SKY_AFTER)) >= 2}
print("2 枚以上あるフィルター:", list(img_t))
show(img_t["Ha"], title="tonight: H-alpha (+[N II]), 60 s");

In [ ]:
# TODO: 今夜の M57 の中心 (xc_t, yc_t) を求める（目で見た中心を目安に centroid(img_t["Ha"], x, y, r=60)）
xc_t, yc_t = ...

rel = [(x - xc, y - yc) for x, y in positions]           # 課題2の開口を、中心からの位置に直す
positions_t = [(xc_t + dx, yc_t + dy) for dx, dy in rel]

S_t, snr1_t = {}, {}
for f in img_t:
    im = img_t[f] - sky_level(img_t[f], xc_t, yc_t)
    vals = [aperture_mean(im, x, y, r=3) for x, y in positions_t]
    S_t[f] = np.mean(vals)
    sig_bg_t = bg_noise(pick("Light", f, 60, after=SKY_AFTER), xc_t, yc_t)
    # TODO: 1 枚・1 画素の S/N（課題2と同じ式。g は課題2で求めたもの）
    snr1_t[f] = ...
    print(f, " S:", S_t[f], "(10/8:", S.get(f), ")   S/N:", snr1_t[f], "(10/8:", snr1_mew.get(f), ")")

### A-3 露出時間を変えた Hα

Hα を 30・60・120 秒で撮っていれば、1 枚の S/N が露出時間とともにどう伸びるかを見ます。ノイズがダーク電流や空の光（露出に比例して増える）で決まっていれば S/N は √t に、読み出しノイズ（露出によらない）で決まっていれば t に比例して伸びます。

星がいちばん明るい画素の値も確かめます。値がある上限に張り付いていたら、飽和しています。

In [ ]:
sky_ha = T[(T.kind == "Light") & (T.filt == "Ha") & ((T.jst >= SKY_AFTER) if SKY_AFTER else True)]
exps_ha = sorted(e for e, g in sky_ha.groupby("exp") if len(g) >= 2)
print("2 枚以上ある Hα の露出時間:", exps_ha)

t_list, snr_list = [], []
for e in exps_ha:
    ps = pick("Light", "Ha", e, after=SKY_AFTER)
    im = stack_list(ps, ref=REF_T)
    im = im - sky_level(im, xc_t, yc_t)
    s_ring = np.mean([aperture_mean(im, x, y, r=3) for x, y in positions_t])   # s（ピクセルスケール）を上書きしないよう別の名前にする
    sig = bg_noise(ps, xc_t, yc_t)
    t_list.append(e)
    snr_list.append(s_ring / np.sqrt(sig**2 + s_ring / g))
    print(f"{e:5.0f} s  S = {s_ring:8.1f} ADU   S/N = {snr_list[-1]:6.1f}   画像の最大値 = {max(load(p).max() for p in ps):.0f} ADU")

# TODO: t_list と snr_list を両対数でプロットし、傾き 1/2（√t）と傾き 1（t）の線と比べる

**答え：**（今夜の値は予測〔10/8 の値〕と比べてどうだったか。違いがあれば、その原因として何が考えられるか。空の明るさ、センサー温度、高度などに触れること）

（ここに書く）

## 夕方B（曇りの場合）　屋内で装置を測る

### B-1 露出時間を変えたダーク

キャップをしたまま、露出時間を変えてダークを 2 枚ずつ撮りました。露出時間ごとに平均レベルと 1 枚のノイズを求め、次の 2 本の直線を当てはめます（g は課題2のゲイン、t は露出時間）。

$$\text{平均レベル} = \text{オフセット} + D\,t, \qquad \sigma^2 = \sigma_{\rm read}^2 + D\,t / g$$

D は ADU/s で表したダーク電流、σ_read は読み出しノイズ（ADU）です。電子の単位に直すと、ダーク電流は D × g（e⁻/s）、読み出しノイズは σ_read × g（e⁻）です。

In [ ]:
D_frames = T[T.kind == "Dark"]
dark_exps = sorted(e for e, g in D_frames.groupby("exp") if len(g) >= 2)
print("2 枚以上あるダークの露出時間:", dark_exps, "  センサー温度:", D_frames.temp.min(), "〜", D_frames.temp.max(), "℃")

d_lev, d_sig = [], []
for e in dark_exps:
    lv, sg = pair_stats(pick("Dark", exp=e))
    d_lev.append(lv); d_sig.append(sg)
    print(f"{e:6.1f} s  level = {lv:8.2f} ADU   sigma = {sg:6.2f} ADU")
d_t, d_lev, d_sig = np.array(dark_exps), np.array(d_lev), np.array(d_sig)

# TODO: d_t に対して d_lev と d_sig**2 をプロットし、np.polyfit で直線を当てはめる
#       → オフセット、D [ADU/s]、σ_read [ADU]、それを電子に直した値

**答え：**（1. 60 秒の露出では、読み出しノイズとダーク電流のノイズのどちらが大きいか。2. 両者が等しくなる露出時間は何秒か。本番の 1 枚の露出を 60 秒にする判断は妥当か。3. 10/8〔センサー約 24℃〕と比べて、今日の温度でダーク電流はどう違ったか）

（ここに書く）

### B-2 一様な光源でゲインと飽和を測る

白い壁などの一様な光源を、同じフィルターで露出時間を変えて 2 枚ずつ撮りました。課題2のゲインは空の背景（暗い範囲）だけで求めましたが、ここでは明るい範囲まで測れます。

画像の中央付近（`box`）で、露出時間ごとに平均レベルと 1 枚のノイズを求め、σ² を平均レベルに対してプロットします。直線の傾きが 1／g です（課題2と同じフォトン・トランスファー・カーブ）。明るくなると直線から外れる（σ² が頭打ちになったり急に下がったりする）ところが、飽和や非線形の始まりです。

撮った時刻で絞り込むと、夜空のフレームと区別できます（`pick(..., after="16:40", before="17:20")` など）。

In [ ]:
FLAT_FILT = "Ha"                       # ← 一様光源を撮ったフィルター
FLAT_AFTER, FLAT_BEFORE = None, None   # ← 撮った時間帯（例: "16:40", "17:20"）
box = (534, 834, 309, 609)             # 画像中央の 300 × 300 画素

L_frames = T[(T.kind == "Light") & (T.filt == FLAT_FILT)]
if FLAT_AFTER:  L_frames = L_frames[L_frames.jst >= FLAT_AFTER]
if FLAT_BEFORE: L_frames = L_frames[L_frames.jst < FLAT_BEFORE]
flat_exps = sorted(e for e, g in L_frames.groupby("exp") if len(g) >= 2)

f_lev, f_sig = [], []
for e in flat_exps:
    lv, sg = pair_stats(pick("Light", FLAT_FILT, e, FLAT_AFTER, FLAT_BEFORE), box)
    f_lev.append(lv); f_sig.append(sg)
    print(f"{e:8.3f} s  level = {lv:9.1f} ADU   sigma = {sg:7.2f} ADU")
f_lev, f_sig = np.array(f_lev), np.array(f_sig)

# TODO: f_lev に対して f_sig**2 をプロットする。直線に乗る範囲だけで当てはめてゲイン g_flat を求め、
#       課題2の g と比べる。直線から外れ始めるレベル（飽和の目安）も読み取る

### B-3 フィルターごとのフラット

同じ一様光源を別のフィルターでも撮っていれば、それぞれを中央値で割って規格化し、2 つのフィルターの比の画像を作ります。周辺減光やゴミの影がフィルターで違えば、比の画像に模様が残ります。

In [ ]:
def flat_image(filt, offset=0.0, after=FLAT_AFTER, before=FLAT_BEFORE):
    """そのフィルターの一様光源のフレームを平均し、offset（ADU）を引いてから中央値で割って規格化する"""
    ps = pick("Light", filt, after=after, before=before)
    im = np.mean([load(p) for p in ps], axis=0) - offset
    return im / np.median(im)

# TODO: 2 つのフィルター（例: "Ha" と "O3"）の規格化フラットを作り、比の画像を show() で表示する
#       offset には B-1 で求めたオフセットを入れる

一様光源が撮れなかった場合は、10/8 の日没前に撮った空のフラット（`masterflat`）で同じことを行います。比の画像で、どの模様が消え、どの模様が残るかを見てください。

In [ ]:
# TODO（一様光源がない場合）: 10/8 のマスターフラットで 2 つのフィルターの比の画像を作り、show() で表示する
#       例: ratio = masterflat("Ha") / masterflat("O3")

**答え：**（ゲインは課題2と一致したか。飽和はおよそ何 ADU から始まるか。フィルターごとにフラットを撮る必要はあるか）

（ここに書く）

## 夕方C（共通）　1 枚あたりにかかる時間

本番のスケジュールは、露出時間だけで組むと足りなくなります。読み出し・保存や、フィルターの交換にも時間がかかるからです。今夜撮ったフレームの撮影開始時刻から、1 枚ごとのオーバーヘッド（前のフレームの開始から次の開始までの時間 − 前のフレームの露出時間）を求めます。

晴れの日はテスト露光、曇りの日は手順の通し稽古（空撮り）のフレームを使います。

In [ ]:
seq = T.copy()
seq["overhead_s"] = (seq.start.diff().dt.total_seconds() - seq.exp.shift()).round(1)
seq["filter_change"] = (seq.filt != seq.filt.shift()) & seq.index.to_series().gt(0)
seq[["jst", "kind", "filt", "exp", "overhead_s", "filter_change"]]

In [ ]:
# TODO: 同じフィルターで続けて撮ったときと、フィルターを替えたときの、オーバーヘッドの典型的な値（中央値）
#       （休憩や待ち時間で間が空いたところは外す。例: overhead_s < 120 のものだけ使う）
oh_same = ...
oh_change = ...
print("同じフィルター:", oh_same, "s   フィルター交換:", oh_change, "s")

**答え：**（課題4で決めた枚数を撮るのに、オーバーヘッドを含めると何分かかるか。課題5のスケジュールに反映すること）

（ここに書く）

## 課題5（宿題）M57 の観測可能時間と当日のスケジュール

11月の M57 は夕方の西空にあり、薄明が終わるとまもなく低くなります。教員が指定する本番日について、使える時間を計算し、課題4の積分時間を割り付けたスケジュールを作ります。

| 計算に使う値 | |
| --- | --- |
| 観測地（国立天文台三鷹） | 北緯 35.672°、東経 139.538°、標高 58 m |
| M57 の位置（J2000） | α = 18h53m35.1s、δ = +33°01′45″ |
| 観測を許す高度の下限 | 30°（エアマス約 2） |

エアマスは X ≈ 1 / sin(高度) で近似できます。

撮影の時間には、夕方C で測ったオーバーヘッドも含めてください。

In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz, get_sun, get_body
from astropy.utils import iers
iers.conf.auto_download = False          # 地球回転パラメータのダウンロードを省略（分単位の計算には影響しない）

OBS_DATE = "2026-11-11"                  # ← 教員が指定する本番日に書き換える
site = EarthLocation(lat=35.672 * u.deg, lon=139.538 * u.deg, height=58 * u.m)
m57 = SkyCoord("18h53m35.1s", "+33d01m45s")

jst_min = np.arange(16 * 60 + 30, 21 * 60 + 31, 1)                       # 16:30〜21:30 を 1 分刻み（日本時間）
times = Time(f"{OBS_DATE} 00:00:00") + (jst_min - 9 * 60) * u.min      # UTC に直す
frame = AltAz(obstime=times, location=site)
sun_alt = get_sun(times).transform_to(frame).alt.deg
m57_alt = m57.transform_to(frame).alt.deg

def hhmm(m):
    return f"{int(m // 60):02d}:{int(m % 60):02d}"
print("計算できました:", OBS_DATE)

In [ ]:
# TODO: 日没（太陽高度 -0.83°）、市民・航海・天文薄明の終了（-6°, -12°, -18°）の時刻
#       ヒント: sun_alt が初めて閾値を下回る位置を np.argmax(sun_alt < 閾値) で探し、jst_min から時刻を読む

# TODO: M57 の高度とエアマスをプロットし、天文薄明終了時の高度と、高度 30° を下回る時刻を求める

月の明るさと位置も確認します（天文薄明終了の時刻で計算）。輝面比は太陽と月の離角 φ から (1 − cos φ) / 2 で求まります。

In [ ]:
# TODO: 天文薄明終了時の月の輝面比・高度・M57 との離角
#       ヒント: moon = get_body("moon", times[i], site); sun = get_sun(times[i])

撮影の順番を決めるときは、次の 2 点を考えてください。

1. 高度が下がると大気減光が増え、減光は短い波長（[O III] 501 nm）ほど大きい
2. 輝線比を求めるには、フィルターごとのエアマスがそろっているほうがよい（交互に撮る方法もある）

フラットは空が明るいうちに、暗いナローバンドから撮ります。天候や機材のトラブルに備えて、予備時間も入れてください。

**答え：**

| 時刻 | 作業 | フィルター | 露出 × 枚数 | 確認すること |
| --- | --- | --- | --- | --- |
| | 機材の準備（カメラを外気になじませ、センサー温度を記録） | | | |
| （日没） | トワイライトフラット | | | |
| | 明るい星でピント合わせ | | | |
| | M57 の導入、試し撮り | | | |
| （天文薄明終了） | 本観測 | | | |
| （高度 30°） | 観測終了 | | | |
| | ダーク（同じ露出・同じ温度） | | | |